**Parse through the reports and organize outputs**

In [1]:
import pandas as pd

# Parameters
path_to_hbs_notes = '/Users/rm026/Documents/hbs_study_patient_notes/'
note_path = 'dis_test_file.txt'
output_reports_path = '/Users/rm026/Documents/hbs_study_patient_notes/dis_notes_test/'
note_type='dis_notes'

separator = '|'
test_limit = None  # process only the first 20 headers
# test_limit = None


In [2]:

# Read and clean the DataFrame
df = pd.read_csv(
    f'{path_to_hbs_notes}/{note_path}', 
    sep=separator, 
    dtype={'EPIC_PMRN': str, 'MRN': str, 'EMPI': str, "Report_Number":str}
)

# Drop rows with all missing values and required missing values
df.dropna(axis=0, how='all', inplace=True)
df.dropna(subset=['EPIC_PMRN', 'MRN'], inplace=True)

# If you need to filter EMPI to numeric values, you can do a regex match rather than casting to int:
df = df[df['EMPI'].str.match(r'^\d+$')].reset_index(drop=True)

# Remove trailing ".0" if present, while keeping any leading zeros intact
df['EPIC_PMRN'] = df['EPIC_PMRN'].str.replace(r'\.0$', '', regex=True)
df['MRN']       = df['MRN'].str.replace(r'\.0$', '', regex=True)
df['EMPI']      = df['EMPI'].str.replace(r'\.0$', '', regex=True)
# df['Report_Number']=[  ]
# df['Report_Number']= df['Report_Number'].str.replace('.0', '',)

# Limit to first test_limit rows
df_test = df.head(test_limit) if test_limit else df

# Read the full text file as lines
with open(f'{path_to_hbs_notes}/{note_path}', 'r') as f:
    file_lines = f.readlines()

# Build header strings for the test set
header_strings = [
    separator.join(row[:-1].astype(str).tolist()) for _, row in df_test.iterrows()]
header_strings=[string+'|' for string in header_strings]


results_list = []
num_headers = len(header_strings)

for i, header in enumerate(header_strings):
    # Find the start index for the current header in file_lines
    start_index = next((j for j, line in enumerate(file_lines) if header in line), None)
    if start_index is None:
        print(f"Warning: Header not found for index {i}")
        print("Used header:", header)
        continue

    # Determine the end index: next header occurrence, or end of file if last header
    if i < num_headers - 1:
        next_header = header_strings[i+1]
        end_index = next(
            (j for j, line in enumerate(file_lines[start_index+1:], start=start_index+1) if next_header in line),
            len(file_lines)
        )
    else:
        end_index = len(file_lines)

    report_text = "".join(file_lines[start_index:end_index]).strip()
    row = df_test.iloc[i]
    results_list.append({
        'MRN': row['MRN'],
        'Report_Number': row['Report_Number'],
        'Report_Description': row['Report_Description'],
        'Report_Date_Time': row['Report_Date_Time'],
        'Report_Text': report_text
    })

# Combine results into a DataFrame using pd.concat
results_extracted = pd.concat([pd.DataFrame([d]) for d in results_list], ignore_index=True)
display(results_extracted)

,MRN,Report_Number,Report_Description,Report_Date_Time,Report_Text
0,4302515,MGHPOE85115297,Discharge Summary,9/27/2010 12:59:00 PM,103507786|10046267364|MGH|4302515|MGHPOE851152...
1,4302515,MGHPOE85114887,Discharge Summary,9/27/2010 12:59:00 PM,103507786|10046267364|MGH|4302515|MGHPOE851148...
2,4302515,1297250,Discharge Summary,9/28/2010 2:16:00 PM,103507786|10046267364|MGH|4302515|1297250|9/28...
3,4225307,MGHPOE37523639,Discharge Summary,11/1/2005 5:00:00 AM,109387240|10025774455|MGH|4225307|MGHPOE375236...
4,4225307,MGHPOE37884133,Discharge Summary,11/14/2005 5:00:00 AM,109387240|10025774455|MGH|4225307|MGHPOE378841...
...,...,...,...,...,...
324,4950251,MGHPOE93957476,Discharge Summary,7/18/2011 9:18:00 PM,106122661|10076980571|MGH|4950251|MGHPOE939574...
325,4950251,MGHPOE93952544,Discharge Summary,7/18/2011 9:18:00 PM,106122661|10076980571|MGH|4950251|MGHPOE939525...
326,4950251,MGHPOE84541003,Discharge Summary,9/4/2010 9:24:00 PM,106122661|10076980571|MGH|4950251|MGHPOE845410...
327,4950251,MGHPOE84449746,Discharge Summary,9/4/2010 9:24:00 PM,106122661|10076980571|MGH|4950251|MGHPOE844497...


**Filter for only included results**

In [3]:
included_mrns_df = pd.read_csv(f'{path_to_hbs_notes}sub_id_to_mrn_included.csv')
# display(included_mrns_df)
results_extracted_filtered = results_extracted[(results_extracted['MRN'].isin(included_mrns_df['MRN_primary'].astype(str))) | (results_extracted['MRN'].isin(included_mrns_df['MRN_alt'].astype(str)))]
results_extracted_filtered['HB ID'] = results_extracted_filtered['MRN'].apply(lambda x: included_mrns_df[included_mrns_df['MRN_primary'].astype(str) == x]['HB ID'].values[0] if x in included_mrns_df['MRN_primary'].astype(str).values else included_mrns_df[included_mrns_df['MRN_alt'].astype(str) == x]['HB ID'].values[0])
results_extracted_filtered

,MRN,Report_Number,Report_Description,Report_Date_Time,Report_Text,HB ID
0,4302515,MGHPOE85115297,Discharge Summary,9/27/2010 12:59:00 PM,103507786|10046267364|MGH|4302515|MGHPOE851152...,31570
1,4302515,MGHPOE85114887,Discharge Summary,9/27/2010 12:59:00 PM,103507786|10046267364|MGH|4302515|MGHPOE851148...,31570
2,4302515,1297250,Discharge Summary,9/28/2010 2:16:00 PM,103507786|10046267364|MGH|4302515|1297250|9/28...,31570
3,4225307,MGHPOE37523639,Discharge Summary,11/1/2005 5:00:00 AM,109387240|10025774455|MGH|4225307|MGHPOE375236...,30981
4,4225307,MGHPOE37884133,Discharge Summary,11/14/2005 5:00:00 AM,109387240|10025774455|MGH|4225307|MGHPOE378841...,30981
...,...,...,...,...,...,...
324,4950251,MGHPOE93957476,Discharge Summary,7/18/2011 9:18:00 PM,106122661|10076980571|MGH|4950251|MGHPOE939574...,31485
325,4950251,MGHPOE93952544,Discharge Summary,7/18/2011 9:18:00 PM,106122661|10076980571|MGH|4950251|MGHPOE939525...,31485
326,4950251,MGHPOE84541003,Discharge Summary,9/4/2010 9:24:00 PM,106122661|10076980571|MGH|4950251|MGHPOE845410...,31485
327,4950251,MGHPOE84449746,Discharge Summary,9/4/2010 9:24:00 PM,106122661|10076980571|MGH|4950251|MGHPOE844497...,31485


**Save as separate .txt files for each patient**

In [4]:

results_extracted_filtered['filepath'] = results_extracted_filtered.apply(lambda x: f'{output_reports_path}sub-{x["HB ID"]}_{note_type}.txt', axis=1)
hb_ids=results_extracted_filtered['HB ID'].unique()
for id in hb_ids:
    notes=results_extracted_filtered[results_extracted_filtered['HB ID']==id]
    filepath = notes.iloc[0]['filepath']
    with open(filepath, 'w') as f:
        for i, row in notes.iterrows():
            f.write(row['Report_Text'])
            f.write('\n')

results_extracted_filtered_no_reports = results_extracted_filtered.drop(columns=['Report_Text'])
results_extracted_filtered_no_reports.to_csv(f'{path_to_hbs_notes}csvs/{note_type}.csv', index=False)